# M04-01 — Joins

Referencia de validación. El alumno trabaja en `notebooks/alumno/M04-01-joins.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
print(fact.count(), customers.count())
inner = fact.join(customers, "customer_id", "inner")
left = fact.join(customers, "customer_id", "left")
orphans = fact.join(customers, "customer_id", "left_anti")
print("inner", inner.count(), "left", left.count())
orphans.select("order_id", "customer_id").distinct().orderBy("order_id").show()
print("líneas", orphans.count(), "pedidos", orphans.select("order_id").distinct().count())
assert fact.count() == 1980 and customers.count() == 250
assert inner.count() == 1956 and left.count() == 1980
assert orphans.count() == 24
assert orphans.select("order_id").distinct().count() == 8
orders = spark.read.parquet(str(STAGING / "orders_clean"))
assert orders.join(customers, "customer_id", "inner").count() == 780
assert orders.join(customers, "customer_id", "left").count() == 788
print("M04-01 OK")
